# 🚀 FastAPI Endpoint Test Notebook

Tests all API endpoints against a **running FastAPI server**.

### Prerequisites
Start the server first:
```bash
uvicorn main:app --reload --port 8000
```

| # | Method | Endpoint | Description |
|---|---|---|---|
| 1 | `GET` | `/` | Health check |
| 2 | `GET` | `/sites/{site_id}` | Fetch site by ID |
| 3 | `GET` | `/sites/nearest/?lat=&lng=` | Nearest site lookup |
| 4 | `POST` | `/score` | Score a single site |
| 5 | `POST` | `/score/what-if` | What-if weight comparison |
| 6 | `POST` | `/compare` | Compare multiple sites |
| 7 | `POST` | `/hotspots` | Find hotspots in a state |

---
## 0. Setup

In [26]:
import httpx
import json
from pprint import pprint

BASE_URL = "http://127.0.0.1:8000"

client = httpx.AsyncClient(base_url=BASE_URL, timeout=60.0)


def show(resp: httpx.Response, label: str = ""):
    """Pretty-print an HTTP response with smart summarization."""
    status_icon = "✅" if resp.status_code < 400 else "❌"
    print(f"\n{status_icon} [{resp.status_code}] {resp.request.method} {resp.request.url}")
    if label:
        print(f"   {label}")
    print(f"   Time: {resp.elapsed.total_seconds():.2f}s")
    try:
        data = resp.json()
        raw = json.dumps(data, indent=2, default=str)
        if len(raw) > 3000:
            # Smart summary for large responses
            if 'ranked_sites' in data:
                print(f"\n   📊 {len(data['ranked_sites'])} sites ranked:")
                for i, s in enumerate(data['ranked_sites'], 1):
                    print(f"     #{i} {s['id']} — score={s['site_readiness_score']:.1f}  ({s['lat']:.3f}, {s['lng']:.3f})")
                if data.get('insight_text'):
                    print(f"\n   🧠 {data['insight_text']}")
            elif 'hotspots' in data:
                print(f"\n   🔥 {len(data['hotspots'])} hotspots:")
                for i, h in enumerate(data['hotspots'], 1):
                    print(f"     #{i} {h['id']} — {h['district']}, {h['state']} — score={h['site_readiness_score']:.1f}")
                if data.get('insight_text'):
                    print(f"\n   🧠 {data['insight_text']}")
            elif 'site_score' in data:
                ss = data['site_score']
                print(f"\n   📊 Site: {ss['id']}  Score: {ss['site_readiness_score']:.1f}/100")
                print(f"   Contributions: {json.dumps(ss['contributions'], indent=6)}")
                if data.get('score_breakdown'):
                    bd = data['score_breakdown']
                    print(f"   Strengths:  {bd.get('strengths', [])}")
                    print(f"   Weaknesses: {bd.get('weaknesses', [])}")
                if data.get('insight_text'):
                    print(f"\n   🧠 {data['insight_text']}")
            else:
                print(raw[:3000])
                print(f"\n   ... ({len(raw):,} chars total, truncated)")
        else:
            print(raw)
    except Exception:
        print(resp.text[:500])
    return resp


print(f"Base URL: {BASE_URL}")
print("Ready — make sure the server is running!")

Base URL: http://127.0.0.1:8000
Ready — make sure the server is running!


---
## 1. `GET /` — Health Check

In [ ]:
resp = await client.get("/")
show(resp, "Health check");

---
## 2. `GET /sites/nearest/` — Find Nearest Site by Coordinates

Uses PostGIS `<->` operator for spatial nearest-neighbor lookup.

In [ ]:
# Ahmedabad coordinates
resp = await client.get("/sites/nearest/", params={"lat": 23.0225, "lng": 72.5714})
r = show(resp, "Nearest site to Ahmedabad")

# Store site_id for later tests
if resp.status_code == 200:
    SITE_ID = resp.json()["id"]
    print(f"\n📌 Stored SITE_ID = {SITE_ID}")

---
## 3. `GET /sites/{site_id}` — Fetch Site by ID

In [ ]:
resp = await client.get(f"/sites/{SITE_ID}")
show(resp, f"Fetch site {SITE_ID}");

In [ ]:
# Test 404 — nonexistent site
resp = await client.get("/sites/IND_9999999")
show(resp, "Expected 404 for nonexistent site");

---
## 4. `POST /score` — Score a Single Site

Runs the **full LangGraph pipeline**:  
`orchestrator → fetch_features → fetch_scores → compute_score → explainability → insight → END`

In [ ]:
# Score with explicit weights
resp = await client.post("/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
})
show(resp, "Score site — Ahmedabad (retail, with weights)");

In [ ]:
# Score without weights (triggers advisory node)
resp = await client.post("/score", json={
    "site_input": {"lat": 19.0760, "lng": 72.8777},
    "use_case": "ev_charging"
})
show(resp, "Score site — Mumbai (EV charging, no weights → advisory)");

---
## 5. `POST /score/what-if` — What-If Weight Comparison

Compares the score under two different weight configurations.

In [ ]:
resp = await client.post("/score/what-if", json={
    "site_id": SITE_ID,
    "current_weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    },
    "modified_weights": {
        "demand_score": 0.10,
        "accessibility_score": 0.10,
        "competition_score": 0.10,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.50
    }
})
show(resp, f"What-if for {SITE_ID} — retail vs infra-heavy weights");

---
## 6. `POST /compare` — Compare Multiple Sites

Compares 3 cities and returns ranked results.

In [27]:
resp = await client.post("/compare", json={
    "sites": [
        {"lat": 23.0225, "lng": 72.5714},   # Ahmedabad
        {"lat": 19.0760, "lng": 72.8777},   # Mumbai
        {"lat": 12.9716, "lng": 77.5946},   # Bangalore
    ],
    "use_case": "warehouse",
    "weights": {
        "demand_score": 0.15,
        "accessibility_score": 0.35,
        "competition_score": 0.05,
        "suitability_score": 0.15,
        "risk_score": 0.15,
        "infrastructure_score": 0.15
    }
})
show(resp, "Compare — Ahmedabad vs Mumbai vs Bangalore (warehouse)");


✅ [200] POST http://127.0.0.1:8000/compare
   Compare — Ahmedabad vs Mumbai vs Bangalore (warehouse)
   Time: 1.59s

   📊 3 sites ranked:
     #1 IND_0012948 — score=86.8  (12.968, 77.612)
     #2 IND_0050492 — score=82.2  (19.088, 72.860)
     #3 IND_0099352 — score=80.5  (23.012, 72.572)

   🧠 Based on the site scores, IND_0012948 is the top-ranked warehouse site with a score of 86.8. It outperforms the other two sites by 4.6 and 6.3 points, respectively. The key differentiator is its high overall score, indicating a strong balance of features. IND_0012948 is the best site due to its significant lead over the others. To maximize warehouse efficiency, I recommend selecting IND_0012948 as the primary site, considering its superior overall score and potential for optimal operations.


---
## 7. `POST /hotspots` — Find Top Hotspot Locations

In [ ]:
resp = await client.post("/hotspots", json={
    "state": "Gujarat",
    "use_case": "retail",
    "top_n": 5
})
show(resp, "Hotspots — Top 5 in Gujarat (retail)");

---
## 8. ❌ Error Cases

In [ ]:
# Invalid use_case
resp = await client.post("/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "invalid_case"
})
show(resp, "Expected 422 — invalid use_case");

In [ ]:
# Weights don't sum to 1.0
resp = await client.post("/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.50,
        "accessibility_score": 0.50,
        "competition_score": 0.50,
        "suitability_score": 0.50,
        "risk_score": 0.50,
        "infrastructure_score": 0.50
    }
})
show(resp, "Expected 422 — weights sum to 3.0");

In [ ]:
# Compare with only 1 site (minimum is 2)
resp = await client.post("/compare", json={
    "sites": [{"lat": 23.0225, "lng": 72.5714}],
    "use_case": "retail"
})
show(resp, "Expected 422 — need at least 2 sites");

---
## 9. ⏱️ Latency Benchmark

Run the scoring endpoint multiple times and measure latency.

In [ ]:
import time

PAYLOAD = {
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
}

N_RUNS = 5
times = []

for i in range(N_RUNS):
    start = time.perf_counter()
    resp = await client.post("/score", json=PAYLOAD)
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"  Run {i+1}/{N_RUNS}: {elapsed:.3f}s {status}")

print(f"\n📊 Latency Summary (POST /score):")
print(f"  Min:  {min(times):.3f}s")
print(f"  Max:  {max(times):.3f}s")
print(f"  Avg:  {sum(times)/len(times):.3f}s")
print(f"  P50:  {sorted(times)[len(times)//2]:.3f}s")

---
## 10. 📋 Swagger Docs Check

In [ ]:
resp = await client.get("/openapi.json")
if resp.status_code == 200:
    spec = resp.json()
    print(f"API Title: {spec['info']['title']}")
    print(f"Version:   {spec['info']['version']}")
    print(f"\nRegistered endpoints:")
    for path, methods in spec["paths"].items():
        for method in methods:
            summary = methods[method].get("summary", "")
            print(f"  {method.upper():6s} {path:35s} — {summary}")
else:
    print(f"Failed to get OpenAPI spec: {resp.status_code}")

---
## 11. 🧹 Cleanup

In [ ]:
await client.aclose()
print("HTTP client closed ✓")